# Gene mention normalization and sentence retrieval

This example uses four constructed sentences, a hand-written alias dictionary and TF-IDF retrieval. It illustrates why domain NLP needs an explicit ontology layer: a correct surface span does not guarantee a correct biological identifier.

In [2]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

abstracts = {
    "p1": "BRCA1 supports homologous recombination repair after DNA double-strand breaks.",
    "p2": "Loss of p53 function permits cells with damaged DNA to continue proliferating.",
    "p3": "Activating mutations in EGFR can predict sensitivity to tyrosine kinase inhibitors.",
    "p4": "The TP53 tumor suppressor induces cell-cycle arrest following genomic stress.",
}
alias_to_gene = {
    "BRCA1": "HGNC:1100",
    "TP53": "HGNC:11998",
    "P53": "HGNC:11998",
    "EGFR": "HGNC:3236",
}

pattern = re.compile(r"\b(" + "|".join(map(re.escape, alias_to_gene)) + r")\b", re.I)
mentions = []
for document_id, text in abstracts.items():
    for match in pattern.finditer(text):
        mentions.append({"document": document_id, "surface": match.group(),
                         "start": match.start(), "end": match.end(),
                         "normalized_id": alias_to_gene[match.group().upper()]})
pd.DataFrame(mentions)

,document,surface,start,end,normalized_id
0,p1,BRCA1,0,5,HGNC:1100
1,p2,p53,8,11,HGNC:11998
2,p3,EGFR,24,28,HGNC:3236
3,p4,TP53,4,8,HGNC:11998


Offsets refer to the original string, before case folding or punctuation changes. This is essential for traceability and for evaluating exact span boundaries.

## Evidence retrieval

In [4]:
document_ids = list(abstracts)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
matrix = vectorizer.fit_transform(abstracts.values())

def search(query, top_k=3):
    scores = cosine_similarity(vectorizer.transform([query]), matrix)[0]
    return pd.DataFrame([
        {"document": document_ids[i], "score": scores[i], "text": abstracts[document_ids[i]]}
        for i in np.argsort(-scores)[:top_k]
    ])

search("Which gene responds to DNA damage by arresting the cell cycle?")

,document,score,text
0,p4,0.381200,The TP53 tumor suppressor induces cell-cycle a...
1,p2,0.115117,Loss of p53 function permits cells with damage...
2,p1,0.062907,BRCA1 supports homologous recombination repair...


## Relation candidates with provenance

The rule below proposes gene–process relations only when both occur in one sentence. Co-occurrence does not establish a relation; precision has not been evaluated on an annotated corpus.

In [6]:
process_terms = {
    "homologous recombination": "GO:0000724",
    "cell-cycle arrest": "GO:0007050",
    "tyrosine kinase inhibitors": "CHEMICAL_CLASS:TKI",
}
relations = []
for document_id, text in abstracts.items():
    genes = [row for row in mentions if row["document"] == document_id]
    for phrase, process_id in process_terms.items():
        if phrase.casefold() in text.casefold():
            for gene in genes:
                relations.append({
                    "document": document_id,
                    "gene_id": gene["normalized_id"],
                    "process_id": process_id,
                    "evidence": text,
                })
pd.DataFrame(relations)

,document,gene_id,process_id,evidence
0,p1,HGNC:1100,GO:0000724,BRCA1 supports homologous recombination repair...
1,p3,HGNC:3236,CHEMICAL_CLASS:TKI,Activating mutations in EGFR can predict sensi...
2,p4,HGNC:11998,GO:0007050,The TP53 tumor suppressor induces cell-cycle a...


## Evaluation design

Span detection needs exact and relaxed boundary F1. Normalization needs identifier accuracy conditional on a correct span. Relation extraction needs labeled entity pairs with direction and negation. Retrieval needs query-level qrels. Reporting these layers separately localizes whether errors come from mention detection, ontology mapping or evidence selection.

The four manually specified gold mentions only check this example. A perfect score here does not estimate performance on research abstracts.

In [8]:
gold_mentions = {
    ("p1", 0, 5, "HGNC:1100"),
    ("p2", 8, 11, "HGNC:11998"),
    ("p3", 24, 28, "HGNC:3236"),
    ("p4", 4, 8, "HGNC:11998"),
}
predicted_mentions = {
    (row["document"], row["start"], row["end"], row["normalized_id"])
    for row in mentions
}
precision = len(gold_mentions & predicted_mentions) / len(predicted_mentions)
recall = len(gold_mentions & predicted_mentions) / len(gold_mentions)
f1 = 2 * precision * recall / (precision + recall)
print({"precision": precision, "recall": recall, "f1": f1})

{'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


Real corpora require species disambiguation, nested mentions, abbreviations and versioned ontology identifiers. A next step would compare PubMedBERT or a biomedical sentence encoder with the lexical baseline under the same qrels and then audit false matches by gene family.